# Inverse Dynamics - Recursive Newton-Euler Algorithm 

The recursions for velocity and accleration allow to compute $\mathcal{V}_i$ and $\dot{\mathcal{V}}_i$ given $\mathbf{q}$, $\dot{\mathbf{q}}$ and $\ddot{\mathbf{q}}$. Recall that inverse dynamics for generalized coordinate $\mathbf{q}$ aim to evaluate the following equation for joint toruqes $\mathbf{\tau}$:

$$
\mathbf{\tau} = \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) 
$$

One can compute $\mathbf{\tau}_i$ at each joint by applying the Newton-Euler equation [](#eq-newtoneuler) in the previous section. As depicted in [](#rnea) each link $L_i$ is subject to joint forces $\mathcal{F}_i$, $\mathcal{F}_{i+1}$ and its own inertial force. The equilibrium equation can be written as:

$$
\begin{split}
& \mathcal{F}_i - \mathcal{F}_{i, i+1} = \mathcal{M}_{i}\dot{\mathcal{V}}_i + \mathcal{V}_i \times^* \mathcal{M}_{i}\mathcal{V}_i \\
\Longrightarrow & \mathcal{F}_i = \mathcal{F}_{i, i+1} + \mathcal{M}_{i}\dot{\mathcal{V}}_i + \mathcal{V}_i \times^* \mathcal{M}_{i}\mathcal{V}_i
\end{split}
$$

which yields another recursive relation but from the outmost link to the basis since $\mathcal{F}_{i}$ depends on knowing $\mathcal{F}_{i+1}$. The recursion begins with $\mathcal{F}_{tip}$ as the force from the external environment applied to the outermost link. Note all quantities must be referenced in the joint frame $i$, e.g. spatial inertia defined at the center-of-mass for $L_i$ need to be transformed if the two frames do not coincide.  

```{figure} ../images/rnea.png
---
name: rnea
---

Spatial forces applied at $L_i$ due to the joint constraint forces from adjacent links. When joint $i+1$ applies $\mathcal{F}_{i+1}$ to $L_{i+1}$, $L_i$ receives $-\mathcal{F}_{i+1}$ at the same reference point according to Newton's 3rd law. 
```

$\mathcal{F}_{i}$ is the combined joint force to keep the link equilibrium. Part of $\mathcal{F}_{i}$ comes from constraining effects of the joint, e.g. a revolute joint will prevent the relative motion between two links along translational and non-rotational axes. The links are free to move along the rotational axis so the remained part must come from the joint torque $\tau_i$, which is simply the projection of $\mathcal{F}_{i}$ in the direction of motion DOF:

$$
\tau_i = \mathcal{F}_{i} \cdot \mathcal{S}_{i}
$$

The overall process for computing $\mathbf{\tau}$ from $\mathbf{q}$, $\dot{\mathbf{q}}$ and $\ddot{\mathbf{q}}$ is hence summarized as:

* Forward pass: from the basis to the outmost link, compute $\mathcal{V}_i$ and $\dot{\mathcal{V}}_i$ using [](#eq-linkvel) and [](#eq-linkacc) in the previous section.
* Backward pass: from the outmost link to the basis, compute $\mathcal{F}_{i}$ and $\tau_i$ using [](#eq-rnea) and [](#eq-rnea_torque). 

This is called __Recursive Newton-Euler Algorithm (RNEA)__ for inverse dynamics computation. It is an efficient method as both the forward and backward passes are $O(N)$ processes so the algorithm complexity is linear with the number of articulated links. 

Let's have a closer look at the recursive relation [](#eq-rnea). Basically the joint torques $\mathbf{\tau}$ is the sum of a term linear with the acceleration (including the gravity in the accleration recursion) and a term quadratic with the velocity. The latter is often referred as [Coriolis](https://en.wikipedia.org/wiki/Coriolis_force) and [centrifugal](https://en.wikipedia.org/wiki/Centrifugal_force) force. In many robotics literature, the bias due to the gravity and the structure with respect to the generalized velocity $\dot{\mathbf{q}}$ are often made explicit, reformulating $\mathbf{\tau} = \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q})$ to:

$$
\mathbf{\tau} = \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) \dot{\mathbf{q}} + \mathbf{g}(\mathbf{q})
$$


# Inversing Mass Matrix for Forward Dynamics

The structure of articulated body dynamics reveals ways to calculate various terms in [](#eq-robodyn) and [](#eq-robodyn_grav). For instance, one can calculate the joint torques required to cancel the gravity effects by setting $\dot{\mathbf{q}}$ and $\ddot{\mathbf{q}}$ to $\mathbf{0}$ in RNEA. This can be used for _gravity compensation_ so the extra torques can control the robot arm as an object floating in the space. 

RNEA can also be utilized to build forward dynamics algorithms. Forward dynamics solve $\ddot{\mathbf{q}}$ given the current $\mathbf{q}$, $\dot{\mathbf{q}}$ and the joint torques $\mathbf{\tau}$. Reviewing the dynamics equation $\mathbf{\tau} = \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q})$, $\ddot{\mathbf{q}}$ can be solved as a linear system when $\mathbf{M}$ and $\mathbf{C}$ are available. It is obvious $\mathbf{C}$ can be obtained by feeding $\ddot{\mathbf{q}} = \mathbf{0}$ to RNEA. For the inertia matrix $\mathbf{M}$, one can evaluate its $i$-th column by feeding $\dot{\mathbf{q}} = \mathbf{0}$ and one-hot $\ddot{\mathbf{q}}$ with the $i$-th entry as one. After assembling $\mathbf{M}$ from the RNEA results, the forward dynamics simply boil down to inversing the inertia matrix $\mathbf{M}$. 

It is worth to note that $\mathbf{M}$ is symmetric and positive-definite, see section 8.4 in [@modernrobotics]. That makes solving the linear system slightly more efficient, e.g. with [Cholesky decomposition](https://en.wikipedia.org/wiki/Cholesky_decomposition). However, it is still with the order of $O(N^3)$ for links with one DOF each. 